# Entrega 1 - Proyecto PENGWIN

Este notebook organiza el primer avance del proyecto: carga del dataset medico en formato `.mha`, verificacion imagen-label, generacion de splits fijos, conversion opcional a NIfTI, ventaneo HU, EDA de fragmentos y visualizacion inicial del hueso.

La prioridad de esta version es la reproducibilidad: el notebook debe correr desde la raiz del repositorio usando rutas relativas y datos locales en `data_mha/`.

## 1. Librerias base

Se importan las librerias necesarias para trabajar con rutas, tablas, volumenes medicos, arreglos numericos y graficas.

In [ ]:
from pathlib import Path
import pandas as pd
import SimpleITK as sitk
import nibabel as nib
import numpy as np
import random
import matplotlib.pyplot as plt

## 2. Configuracion de rutas reproducibles

Todas las rutas se construyen desde `Path.cwd()`, es decir, desde la raiz del repositorio. Esto evita rutas absolutas de un computador especifico y permite que cualquier integrante ejecute el notebook si conserva la estructura esperada.

In [ ]:
BASE_DIR = Path.cwd()

DATA_MHA_DIR = BASE_DIR / "data_mha"

IMAGE_DIRS = [
    DATA_MHA_DIR / "PENGWIN_CT_train_images_part1",
    DATA_MHA_DIR / "PENGWIN_CT_train_images_part2",
]

LABELS_DIR = DATA_MHA_DIR / "PENGWIN_CT_train_labels"
LABEL_DIR = LABELS_DIR  # Alias para compatibilidad con celdas anteriores
SPLITS_DIR = BASE_DIR / "splits"
GENERATED_DATA_DIR = BASE_DIR / "data"

SPLITS_DIR.mkdir(exist_ok=True)
GENERATED_DATA_DIR.mkdir(exist_ok=True)


def to_repo_relative(path):
    """Guarda rutas portables dentro del repositorio."""
    path = Path(path)
    try:
        return path.relative_to(BASE_DIR).as_posix()
    except ValueError:
        return path.as_posix()


def resolve_repo_path(path):
    """Permite leer rutas relativas o absolutas sin romper compatibilidad."""
    path = Path(path)
    if path.is_absolute():
        return path
    return BASE_DIR / path


## 3. Conteo inicial de archivos `.mha`

Se buscan las imagenes CT y las mascaras/labels locales. En esta entrega los datos pesados no se suben a GitHub; cada integrante debe tenerlos en `data_mha/`.

In [ ]:
image_files = []

for image_dir in IMAGE_DIRS:
    image_files.extend(image_dir.glob("*.mha"))

image_files = sorted(image_files)
label_files = sorted(LABELS_DIR.glob("*.mha"))

print("RESUMEN DEL DATASET")


print(f"Imágenes encontradas: {len(image_files)}")
print(f"Labels encontrados:   {len(label_files)}")

## 4. Extraccion de IDs de casos

Se extraen los IDs desde los nombres de archivo para comparar imagenes y labels sin depender de rutas completas.

In [ ]:
image_ids = {
    file.stem
    for file in image_files
}

label_ids = {
    file.stem
    for file in label_files
}

## 5. Validacion de correspondencia imagen-label

Se comprueba que cada imagen tenga su label y que no existan labels sin imagen. Esta validacion es la base del dataset curado.

In [ ]:
images_without_labels = sorted(image_ids - label_ids)
labels_without_images = sorted(label_ids - image_ids)

common_ids = sorted(image_ids & label_ids)

print("VALIDACIÓN DE CORRESPONDENCIA")

print(f"Casos con imagen + label: {len(common_ids)}")

print(
    f"Imágenes sin label:       {len(images_without_labels)}"
)

print(
    f"Labels sin imagen:        {len(labels_without_images)}"
)


if images_without_labels:
    print("\nImágenes sin label:")
    for case_id in images_without_labels:
        print(f"  - {case_id}")


if labels_without_images:
    print("\nLabels sin imagen:")
    for case_id in labels_without_images:
        print(f"  - {case_id}")


## 6. Construccion del dataset curado

Se crea una tabla con `case_id`, ruta relativa de imagen y ruta relativa de label. Las rutas se guardan relativas al repositorio para mantener reproducibilidad entre computadores.

In [ ]:
dataset = []

for case_id in common_ids:

    # Buscar la imagen en todas las partes
    image_path = None

    for image_dir in IMAGE_DIRS:
        possible_path = image_dir / f"{case_id}.mha"

        if possible_path.exists():
            image_path = possible_path
            break

    # Ruta del label
    label_path = LABELS_DIR / f"{case_id}.mha"

    # Verificar que ambos existan
    if image_path is not None and label_path.exists():

        dataset.append({
            "case_id": case_id,
            "image_path": to_repo_relative(image_path),
            "label_path": to_repo_relative(label_path)
        })


df = pd.DataFrame(dataset)
df


## 7. Exportacion de `dataset.csv`

Se guarda el dataset curado en `splits/dataset.csv`. Este archivo queda versionado en GitHub porque contiene rutas ligeras y reproducibles, no datos pesados.

In [ ]:
dataset_path = SPLITS_DIR / "dataset.csv"

df.to_csv(
    dataset_path,
    index=False
)

print()
print(f"Dataset curado guardado en:")
print(dataset_path)

## 8. Verificacion tecnica de volumenes `.mha`

Se cargan imagen y label con SimpleITK para verificar dimensiones, spacing, origen y direccion. Esto confirma que las mascaras corresponden fisicamente con las tomografias.

In [ ]:
records = []

for _, row in df.iterrows():

    image = sitk.ReadImage(str(resolve_repo_path(row["image_path"])))
    label = sitk.ReadImage(str(resolve_repo_path(row["label_path"])))

    records.append({
        "case_id": row["case_id"],

        # Imagen
        "image_dimension": image.GetDimension(),
        "image_size": image.GetSize(),
        "image_spacing": image.GetSpacing(),

        # Label
        "label_dimension": label.GetDimension(),
        "label_size": label.GetSize(),
        "label_spacing": label.GetSpacing(),

        # Verificaciones
        "same_size": image.GetSize() == label.GetSize(),
        "same_spacing": image.GetSpacing() == label.GetSpacing(),
        "same_direction": image.GetDirection() == label.GetDirection(),
        "same_origin": image.GetOrigin() == label.GetOrigin()
    })

verification_df = pd.DataFrame(records)

verification_df.head(20)


## 9. Resumen de consistencia del dataset

Se resumen las verificaciones anteriores para confirmar que el dataset local esta completo y alineado.

In [ ]:
print("Casos:", len(verification_df))

print("\nDimensiones de las imágenes:")
print(verification_df["image_dimension"].value_counts())

print("\nDimensiones de los labels:")
print(verification_df["label_dimension"].value_counts())

print("\nTamaños CT-label iguales:")
print(verification_df["same_size"].value_counts())

print("\nSpacing CT-label iguales:")
print(verification_df["same_spacing"].value_counts())

### mha -> NIfTI (nii.gz)

## 10. Conversion opcional de `.mha` a NIfTI

El flujo principal trabaja con `.mha`, como recomendo el profesor Ferro. Esta seccion conserva una conversion opcional a `.nii.gz` para analisis con `nibabel`, pero no debe entenderse como requisito central del proyecto.

In [ ]:
NIFTI_IMAGES_DIR = GENERATED_DATA_DIR / "nifti_images"
NIFTI_LABELS_DIR = GENERATED_DATA_DIR / "nifti_labels"

NIFTI_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
NIFTI_LABELS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de imagenes NIfTI:", to_repo_relative(NIFTI_IMAGES_DIR))
print("Carpeta de labels NIfTI:", to_repo_relative(NIFTI_LABELS_DIR))


## 11. Generacion local de archivos NIfTI opcionales

Los NIfTI se generan dentro de `data/nifti_images/` y `data/nifti_labels/`. Estas salidas son locales e ignoradas por Git porque pueden ser pesadas y se pueden regenerar.

In [ ]:
conversion_records = []
convertidos = 0
saltados = 0

for _, row in df.iterrows():

    case_id = row["case_id"]

    image_mha = resolve_repo_path(row["image_path"])
    label_mha = resolve_repo_path(row["label_path"])

    image_nii = NIFTI_IMAGES_DIR / f"{case_id}.nii.gz"
    label_nii = NIFTI_LABELS_DIR / f"{case_id}.nii.gz"

    # Si ambos archivos NIfTI ya existen, no los regeneres
    if image_nii.exists() and label_nii.exists():
        saltados += 1
    else:
        image = sitk.ReadImage(str(image_mha))
        label = sitk.ReadImage(str(label_mha))

        sitk.WriteImage(image, str(image_nii), useCompression=True)
        sitk.WriteImage(label, str(label_nii), useCompression=True)
        convertidos += 1

    conversion_records.append({
        "case_id": case_id,
        "image_path": to_repo_relative(image_nii),
        "label_path": to_repo_relative(label_nii)
    })

nifti_df = pd.DataFrame(conversion_records)

print(f"Convertidos ahora: {convertidos}")
print(f"Ya existian (saltados): {saltados}")
print(f"Total en nifti_df: {len(nifti_df)}")
print(nifti_df.head())


## 12. Verificacion de los NIfTI generados

Se confirma que la conversion opcional conserve dimensiones y spacing entre imagen y label.

In [ ]:
verification_nifti = []

for _, row in nifti_df.iterrows():

    image = sitk.ReadImage(str(resolve_repo_path(row["image_path"])))
    label = sitk.ReadImage(str(resolve_repo_path(row["label_path"])))

    verification_nifti.append({
        "case_id": row["case_id"],
        "image_dimension": image.GetDimension(),
        "label_dimension": label.GetDimension(),
        "image_size": image.GetSize(),
        "label_size": label.GetSize(),
        "image_spacing": image.GetSpacing(),
        "label_spacing": label.GetSpacing(),
        "same_size": image.GetSize() == label.GetSize(),
        "same_spacing": image.GetSpacing() == label.GetSpacing()
    })

nifti_verification_df = pd.DataFrame(verification_nifti)

print(nifti_verification_df.head())


## 13. Resumen de verificacion NIfTI

Se imprimen conteos de dimensiones, tamanos y spacing para revisar rapidamente si la conversion fue consistente.

In [ ]:
print("Dimensiones de imágenes:")
print(nifti_verification_df["image_dimension"].value_counts())

print("\nDimensiones de labels:")
print(nifti_verification_df["label_dimension"].value_counts())

print("\nTamaños iguales:")
print(nifti_verification_df["same_size"].value_counts())

print("\nSpacing iguales:")
print(nifti_verification_df["same_spacing"].value_counts())

## 14. Registro local del dataset NIfTI

Se guarda un CSV local con las rutas NIfTI generadas. Este archivo no se sube a GitHub porque es una salida reproducible del notebook.

In [ ]:
DATASET_NIFTI_CSV = GENERATED_DATA_DIR / "dataset_nifti.csv"

nifti_df.to_csv(DATASET_NIFTI_CSV, index=False)

print("Dataset NIfTI guardado en:")
print(to_repo_relative(DATASET_NIFTI_CSV))


## 15. Conteo inicial de fragmentos por caso

A partir de las labels se cuenta cuantas etiquetas diferentes aparecen en cada caso, excluyendo el fondo. Esto da una primera idea de la complejidad del dataset.

In [ ]:
frag_counts = []

for _, row in nifti_df.iterrows():
    label_img = nib.load(str(resolve_repo_path(row["label_path"])))
    label_vol = label_img.get_fdata()
    n_fragments = len(np.unique(label_vol)) - 1  # -1 por excluir el fondo (label 0)
    frag_counts.append(n_fragments)

nifti_df["n_fragments"] = frag_counts

print("Distribucion de numero de fragmentos por caso:")
print(nifti_df["n_fragments"].describe())
print("\nConteo de valores (cuantos casos tienen cada n de fragmentos):")
print(nifti_df["n_fragments"].value_counts().sort_index())


## 16. Splits fijos recomendados por asesoria

Despues de la asesoria con Ferro, se usa una particion 75/5/20: 75 casos para entrenamiento, 5 para validacion y 20 para test. La semilla fija permite reproducir siempre los mismos splits.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42


def complexity_bin(n_fragments):
    if n_fragments <= 4:
        return "baja"
    if n_fragments <= 6:
        return "media"
    return "alta"

split_source = nifti_df[["case_id", "image_path", "label_path", "n_fragments"]].copy()
split_source["complexity"] = split_source["n_fragments"].apply(complexity_bin)

train_val_df, test_df = train_test_split(
    split_source,
    test_size=20,
    random_state=SEED,
    shuffle=True,
    stratify=split_source["complexity"],
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=5,
    random_state=SEED,
    shuffle=True,
    stratify=train_val_df["complexity"],
)

splits = {
    "train": train_df.sort_values("case_id"),
    "val": val_df.sort_values("case_id"),
    "test": test_df.sort_values("case_id"),
}

for name, split_df in splits.items():
    (SPLITS_DIR / f"{name}.txt").write_text("\n".join(split_df["case_id"].tolist()) + "\n")

split_dataset = []
for name, split_df in splits.items():
    temp = split_df.copy()
    temp.insert(1, "split", name)
    split_dataset.append(temp)

split_dataset = pd.concat(split_dataset, ignore_index=True).sort_values("case_id")
split_dataset.to_csv(SPLITS_DIR / "dataset.csv", index=False)

print("Splits fijos con semilla:", SEED)
print(split_dataset["split"].value_counts().reindex(["train", "val", "test"]))
print("\nDistribucion por complejidad:")
print(split_dataset.groupby(["split", "complexity"]).size().unstack(fill_value=0))


## 17. Ventaneo HU para visualizacion de hueso

Se define una ventana osea aproximada usando centro 400 HU y ancho 1800 HU. El objetivo es redistribuir intensidades para resaltar hueso y justificar el preprocesamiento con valores HU.

In [ ]:
def load_nifti(path):
    img = nib.load(str(resolve_repo_path(path)))
    volume = img.get_fdata()
    spacing = img.header.get_zooms()  # (x, y, z) en mm
    return volume, spacing


def apply_hu_window(volume, window_center=400, window_width=1800):
    low = window_center - window_width / 2
    high = window_center + window_width / 2
    windowed = np.clip(volume, low, high)
    windowed = (windowed - low) / (high - low)
    return windowed

# prueba rapida sobre un caso
volume, spacing = load_nifti(nifti_df.loc[0, "image_path"])
windowed = apply_hu_window(volume)
print(f"Caso {nifti_df.loc[0, 'case_id']}: spacing={spacing}, "
      f"rango original=({volume.min():.1f}, {volume.max():.1f}), "
      f"rango ventaneado=({windowed.min():.2f}, {windowed.max():.2f})")


## 18. Comparacion visual antes y despues del ventaneo

Se muestra un corte del caso 001 sin ventanear y con ventana osea. Esta evidencia ayuda a explicar por que el ventaneo mejora la discriminacion visual del hueso.

In [ ]:
case_id = "001"
row = nifti_df[nifti_df["case_id"] == case_id].iloc[0]

volume, spacing = load_nifti(row["image_path"])
windowed = apply_hu_window(volume)

# corte del medio del volumen (eje Z)
mid_slice = volume.shape[2] // 2  # con nibabel el eje Z suele ser el último

fig, axes = plt.subplots(1, 2, figsize=(10, 10))

axes[0].imshow(volume[:, :, mid_slice], cmap="gray")
axes[0].set_title(f"Caso {case_id} — sin ventanear (HU crudo)")
axes[0].axis("off")

axes[1].imshow(windowed[:, :, mid_slice], cmap="gray")
axes[1].set_title(f"Caso {case_id} — con ventana ósea")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 19. EDA de fragmentos por region anatomica

Se asigna cada label a una region anatomica aproximada: sacro, coxal izquierdo o coxal derecho. Luego se calcula el volumen fisico de cada fragmento usando el spacing en milimetros.

In [ ]:
def label_to_region(label):
    if label == 0:
        return "background"
    elif 1 <= label <= 10:
        return "sacro"
    elif 11 <= label <= 20:
        return "coxal_izquierdo"
    elif 21 <= label <= 30:
        return "coxal_derecho"
    else:
        return "desconocido"

fragment_rows = []

for _, row in nifti_df.iterrows():
    label_img = nib.load(str(resolve_repo_path(row["label_path"])))
    label_vol = label_img.get_fdata()
    spacing = tuple(float(x) for x in label_img.header.get_zooms())
    voxel_volume_mm3 = spacing[0] * spacing[1] * spacing[2]

    labels_present = np.unique(label_vol)
    labels_present = labels_present[labels_present != 0]  # excluir fondo

    for lab in labels_present:
        n_voxels = int(np.sum(label_vol == lab))
        fragment_rows.append({
            "case_id": row["case_id"],
            "label": int(lab),
            "region": label_to_region(int(lab)),
            "n_voxels": n_voxels,
            "volume_mm3": n_voxels * voxel_volume_mm3,
        })

frag_df = pd.DataFrame(fragment_rows)
frag_df.to_csv(GENERATED_DATA_DIR / "metadata_fragments.csv", index=False)
print(f"Total de fragmentos individuales en el dataset: {len(frag_df)}")
frag_df.head(10)


## 20. Resumen general del EDA

Se reporta el numero total de fragmentos y su distribucion por caso. Esto permite identificar si hay casos simples y casos mas complejos.

In [ ]:
print("=== RESUMEN GENERAL ===")
print(f"Total de fragmentos (todas las regiones, todos los casos): {len(frag_df)}")
print(f"Fragmentos por caso -> min: {frag_df.groupby('case_id').size().min()}, "
      f"max: {frag_df.groupby('case_id').size().max()}, "
      f"media: {frag_df.groupby('case_id').size().mean():.2f}")

print("\n=== DISTRIBUCIÓN POR REGIÓN ANATÓMICA ===")
print(frag_df["region"].value_counts())

print("\n=== FRAGMENTOS POR REGIÓN, POR CASO (promedio) ===")
frag_por_region_caso = frag_df.groupby(["case_id", "region"]).size().reset_index(name="n_fragmentos")
print(frag_por_region_caso.groupby("region")["n_fragmentos"].describe())

## 21. Analisis de volumen de fragmentos

Se revisa la distribucion de volumenes en mm3. Los fragmentos muy pequenos pueden representar retos para segmentacion y metricas.

In [ ]:
print("=== VOLUMEN DE FRAGMENTOS (mm³) ===")
print(frag_df["volume_mm3"].describe())

# Fragmentos sospechosamente pequeños (posible ruido de anotación vs. fragmento real)
umbral_pequeno = frag_df["volume_mm3"].quantile(0.05)  # percentil 5 más chico
pequenos = frag_df[frag_df["volume_mm3"] < umbral_pequeno]
print(f"\nFragmentos por debajo del percentil 5 de volumen (<{umbral_pequeno:.1f} mm³): {len(pequenos)}")
print(pequenos[["case_id", "region", "label", "volume_mm3"]].sort_values("volume_mm3").head(10))

## 22. Fragmento principal por hueso

Para cada caso y region, se identifica el fragmento de mayor volumen como fragmento principal. Los demas fragmentos se consideran conminutos para analisis posterior.

In [ ]:
def get_main_fragment_per_bone(frag_df):
    rows = []
    for (case_id, region), group in frag_df.groupby(["case_id", "region"]):
        if region == "background":
            continue
        main_frag = group.loc[group["volume_mm3"].idxmax()]
        rows.append({
            "case_id": case_id,
            "region": region,
            "main_label": main_frag["label"],
            "main_volume_mm3": main_frag["volume_mm3"],
            "n_fragmentos_totales": len(group),
            "n_fragmentos_conminutos": len(group) - 1,
        })
    return pd.DataFrame(rows)

main_frag_df = get_main_fragment_per_bone(frag_df)
main_frag_df.to_csv(GENERATED_DATA_DIR / "main_fragments.csv", index=False)
main_frag_df.head(10)


## 23. Huesos fracturados y no fracturados por region

Se resume cuantos huesos tienen un solo fragmento y cuantos presentan multiples fragmentos por region anatomica.

In [ ]:
print("=== HUESOS SIN FRACTURA (un solo fragmento) POR REGIÓN ===")
sin_fractura = main_frag_df[main_frag_df["n_fragmentos_totales"] == 1]
print(sin_fractura["region"].value_counts())

print("\n=== % de huesos fracturados por región ===")
total_por_region = main_frag_df["region"].value_counts()
fracturados_por_region = main_frag_df[main_frag_df["n_fragmentos_totales"] > 1]["region"].value_counts()
pct_fracturado = (fracturados_por_region / total_por_region * 100).round(1)
print(pct_fracturado)

## 24. Graficas exploratorias del EDA

Se generan graficas para visualizar fragmentos por caso, volumen por region y distribuciones generales del dataset.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Histograma: n° de fragmentos por caso (a nivel dataset completo)
frag_por_caso = frag_df.groupby("case_id").size()
axes[0, 0].hist(frag_por_caso, bins=range(frag_por_caso.min(), frag_por_caso.max() + 2), 
                 edgecolor="black", color="steelblue")
axes[0, 0].set_title("Fragmentos totales por caso")
axes[0, 0].set_xlabel("N° de fragmentos")
axes[0, 0].set_ylabel("N° de casos")

# Boxplot: volumen de fragmento por región
frag_df.boxplot(column="volume_mm3", by="region", ax=axes[0, 1])
axes[0, 1].set_title("Volumen de fragmentos por región (mm³)")
axes[0, 1].set_yscale("log")  # log porque probablemente hay outliers grandes
plt.sca(axes[0, 1])
plt.xticks(rotation=15)

# Barras: cantidad de fragmentos por región (total dataset)
frag_df["region"].value_counts().plot(kind="bar", ax=axes[1, 0], color="coral")
axes[1, 0].set_title("Total de fragmentos por región anatómica")
axes[1, 0].set_ylabel("N° de fragmentos")

# Barras: % de huesos fracturados por región
pct_fracturado.plot(kind="bar", ax=axes[1, 1], color="seagreen")
axes[1, 1].set_title("% de huesos fracturados por región")
axes[1, 1].set_ylabel("%")
axes[1, 1].set_ylim(0, 100)

plt.suptitle("")  # quita el título automático feo que pone pandas
plt.tight_layout()
plt.show()

## 25. Visualizacion volumetrica inicial

A partir de aqui se construye la evidencia visual del volumen crudo. La version historica usa MIP y nube de puntos; la mejora recomendada por Ferro queda implementada en el script de malla `src/visualization/generar_visualizador1_mesh_raw.py`.

## 25. Visualizador 1: MIP raw

Se define una proyeccion de maxima intensidad usando un umbral HU para hueso. El MIP conserva, para cada direccion, los voxeles de mayor intensidad y permite una vista volumetrica inicial.

In [ ]:
def compute_mip(volume, hu_threshold=300, axis=2):
    """
    Proyección de máxima intensidad, mostrando solo hueso (HU > umbral).
    axis=2 -> proyección axial (mirando desde arriba/abajo, eje Z con nibabel)
    axis=1 -> proyección coronal (mirando de frente)
    axis=0 -> proyección sagital (mirando de lado)
    """
    bone_only = np.where(volume > hu_threshold, volume, 0)

    mip = np.max(bone_only, axis=axis)
    return mip

## 26. Evidencia MIP en tres vistas

Se generan vistas axial, coronal y sagital del caso 001. Esta evidencia complementa el visualizador 3D y ayuda a documentar el volumen crudo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

mip_axial = compute_mip(volume, hu_threshold=300, axis=2)
mip_coronal = compute_mip(volume, hu_threshold=300, axis=1)
mip_sagital = compute_mip(volume, hu_threshold=300, axis=0)

# vmax limitado al rango típico de hueso, NO al máximo real del volumen (que incluye metal)
vmax_hueso = 1500

axes[0].imshow(mip_axial.T, cmap="gray", origin="lower", vmin=300, vmax=vmax_hueso)
axes[0].set_title("MIP Axial")
axes[0].axis("off")

axes[1].imshow(mip_coronal.T, cmap="gray", origin="lower", vmin=300, vmax=vmax_hueso)
axes[1].set_title("MIP Coronal")
axes[1].axis("off")

axes[2].imshow(mip_sagital.T, cmap="gray", origin="lower", vmin=300, vmax=vmax_hueso)
axes[2].set_title("MIP Sagital")
axes[2].axis("off")

plt.suptitle(f"Caso {case_id} — MIP raw (umbral HU=300, vmax={vmax_hueso})")
plt.tight_layout()
plt.show()